# RLA-WM Colab demo (PushT)

Two sections:
1. Load the released PushT RLA-WM and run a single t→t+h flow rollout.
2. Train PushT WMRL for 50 iterations on a T4 (seed 7).

Runtime: select **GPU (T4)** under *Runtime → Change runtime type*. Budget ≈ 45 min including downloads.


## Setup


In [ ]:
import os, sys, subprocess, pathlib

REPO_URL  = "https://github.com/mlzxy/rla-wm.git"   
REPO_PATH = "/content/rla-wm"

if not pathlib.Path(REPO_PATH, "train.py").exists():
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_PATH], check=True)

os.chdir(REPO_PATH)
sys.path[:0] = [REPO_PATH, os.path.join(REPO_PATH, "third_party", "diffusion_policy")]
print("cwd:", os.getcwd())


In [ ]:
%pip install -q --upgrade pip
%pip install -q \
    "mani-skill>=3.0.0b22" \
    "open-clip-torch>=3.2.0" "transformers>=4.57.3" "diffusers>=0.21.4" \
    "huggingface-hub>=0.36.0" \
    "omegaconf>=2.3.0" "easydict" "ruamel-yaml>=0.18.16" "h5py>=3.15.1" \
    "imageio>=2.37.2" "imageio-ffmpeg>=0.6.0" "opencv-python-headless>=4.12.0.88" \
    "peft>=0.19.0" "roma>=1.5.4" "scipy>=1.15.3" "tensorboard>=2.20.0" \
    "rich" "tyro" "tqdm" "pandas" "wandb" "decord" "jaxtyping" "jstyleson" "redis" "lpips" \
    "git+https://github.com/EasternJournalist/utils3d"


In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

snapshot_download(
    repo_id="xyzhang368/RLA-WM",
    repo_type="model",
    allow_patterns=[
        "rla/maniskill/**",
        "rla-wm/maniskill/ur10e/**",
        "dino-to-image_unet/maniskill/**",
        "wmrl_checkpoints/0_bc_s2r_nstate/**",
    ],
    local_dir="runs/weights",
)
print("weights ready:", sorted(os.listdir("runs/weights")))


In [ ]:
import tarfile

tar_path = hf_hub_download(
    repo_id="xyzhang368/RLA-WM",
    repo_type="dataset",
    filename="maniskill_pusht.tar",
    local_dir="data",
)

with tarfile.open(tar_path) as t:
    t.extractall(path="data")

os.remove(tar_path) 
n_traj = len(os.listdir("data/maniskill/ppo/ur10e_stick/PushT-v2/success"))
print(f"PushT trajectories ready: {n_traj}")


## §1 — RLA-WM inference (t → t+h)

Loads the released RLA-WM for the **UR10e-stick / PushT** combo, picks one successful trajectory, and predicts the future frame at horizon `h` from the current frame conditioned on the trajectory's `target_qpos`.


In [ ]:
%cd ..
import os, sys
import torch
import numpy as np
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

from utils.misc import load_config
import src.models as models
import src.datasets as datasets
from src.trainers.rla_wm_trainer import RLAWMTrainer
from src.utils.loss_utils import ssim as compute_ssim, lpips as compute_lpips

RLA_WORKDIR      = "runs/weights/rla/maniskill/20260323_21-41-25"
RLAWM_CONFIG     = "configs/rla_wm/ur10e.yaml"
RLAWM_WORKDIR    = "runs/weights/rla-wm/maniskill/ur10e/20260404_15-49-19"
IMG_DECODER_CKPT = "runs/weights/dino-to-image_unet/maniskill/20260404_22-53-36"

DEVICE      = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
EULER_STEPS = 30
ROBOT_ID    = 2     # 0=panda, 1=xarm, 2=ur10e_stick
TASK_IND    = 0     # 0=PushT-v2 (per src/datasets/trajectory_dataset.py TASKS)
HORIZON     = 15
SAMPLE_SEED = 7

rlawm_cfg = load_config(RLAWM_CONFIG)

rlawm_models = {
    name: getattr(models, m["name"])(**m["args"]).to(DEVICE)
    for name, m in rlawm_cfg["models"].items()
}
trainer_kw = dict(rlawm_cfg["trainer"]["args"])
trainer_kw.pop("load_dir", None)
trainer_kw["batch_size"]              = 1
trainer_kw["num_workers"]             = 0
trainer_kw["skip_initial_snapshot"]   = True
trainer_kw["image_decoder_ckpt"]      = IMG_DECODER_CKPT
trainer_kw["encoder_ckpt"]            = RLA_WORKDIR

rlawm_trainer = RLAWMTrainer(
    rlawm_models, None,
    output_dir=RLAWM_WORKDIR, load_dir=RLAWM_WORKDIR, step=None,
    inference_only=True, **trainer_kw,
)
for m in rlawm_trainer.models.values():
    m.eval()
print("loaded:", list(rlawm_trainer.models))


In [ ]:
# Load a small dataset slice and pick the first successful PushT episode.
ds_args = dict(rlawm_cfg["dataset"]["args"])
ds_args["manual_limit"] = 64
ds_args["trajectory_info_cache_file"] = "runs/colab/cache/colab_demo_dataset.json"
dataset = datasets.TrajectoryDataset(**ds_args)

torch.manual_seed(SAMPLE_SEED); np.random.seed(SAMPLE_SEED)
sample = next(
    (dataset[i] for i in range(len(dataset))
     if bool(np.asarray(dataset[i]["success"]).reshape(-1)[0])),
    None,
)
if sample is None:
    raise RuntimeError("No successful PushT trajectory found in scan; raise manual_limit.")

rgbs   = torch.from_numpy(np.asarray(sample["rgbs"])).to(DEVICE)
masks  = torch.from_numpy(np.asarray(sample["foreground_masks"])).to(DEVICE)
masked = rgbs.float() * (masks > 0).float().unsqueeze(2) / 255.0   # (T, 1, 3, H, W) in [0,1]
target_qpos = torch.from_numpy(np.asarray(sample["target_qpos"])).to(DEVICE).float()

with torch.no_grad():
    wm_tokens, patch_hw = rlawm_trainer._extract_dino_tokens(masked[:, :1])
    x_t = wm_tokens[0].float()

    robot_ids = torch.tensor([ROBOT_ID], device=DEVICE)
    task_inds = torch.tensor([TASK_IND], device=DEVICE)
    horizons  = torch.tensor([HORIZON], device=DEVICE, dtype=torch.float)
    full_qpos = rlawm_trainer._target_qpos_to_full_qpos([target_qpos], robot_ids, task_inds)

    flow_model = rlawm_trainer.models["flow_model"]
    num_lat    = getattr(flow_model, "num_tokens", rlawm_trainer.models["encoder"].num_tokens)
    token_dim  = flow_model.token_dim

    gen   = torch.Generator(device=DEVICE).manual_seed(SAMPLE_SEED)
    noise = torch.randn(1, num_lat, token_dim, device=DEVICE, generator=gen)

    sampled = rlawm_trainer._euler_sample(
        flow_model=flow_model, noise=noise, xt_tokens=x_t,
        task_inds=task_inds, horizons=horizons, robot_ids=robot_ids,
        full_qpos_list=full_qpos, steps=EULER_STEPS,
    )
    pred_latent = rlawm_trainer._denormalize_latent_tokens(sampled, flow_model=flow_model)
    _, pred_x_T = rlawm_trainer.models["decoder"](x_t, tokens=pred_latent)
    pred_img    = rlawm_trainer.models["image_decoder"](
        pred_x_T.unsqueeze(1).contiguous(), patch_hw=patch_hw,
    )[:, 0].clamp(0, 1)

gt_t_pil    = TF.to_pil_image(masked[0, 0].clamp(0, 1).cpu())
gt_T_pil    = TF.to_pil_image(masked[1, 0].clamp(0, 1).cpu())
pred_T_pil  = TF.to_pil_image(pred_img[0].clamp(0, 1).cpu())

lpips_v = compute_lpips(pred_img, masked[1, :1]).item()
ssim_v  = compute_ssim(pred_img, masked[1, :1]).item()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, ttl in zip(
    axes,
    [gt_t_pil, gt_T_pil, pred_T_pil],
    ["frame t (input)", f"frame t+{HORIZON} (GT)",
     f"frame t+{HORIZON} (RLA-WM pred)\nLPIPS={lpips_v:.3f}  SSIM={ssim_v:.3f}"],
):
    ax.imshow(img); ax.set_title(ttl, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
import gc
del rlawm_trainer, rlawm_models, flow_model, sampled, pred_latent, pred_x_T, pred_img, wm_tokens
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"free GPU memory: {free / 1e9:.1f} / {total / 1e9:.1f} GB")


## §2 — WMRL training 

Hyperparameters tuned for a free-tier T4 (16 GB):

- `num_envs=32`, `env_batch_num=4` → world-model forward pass runs as 4 sequential chunks of 8 envs each (single-process; `MultiProcessWorldModelVecEnv` only triggers on multi-GPU).
- `num_steps=4`, `mini_batch_size=64`, `update_epochs=4` → 32×4/64 = 2 SGD steps per epoch, ×4 epochs = 8 updates per rollout.
- `total_iterations=50`, `eval_freq=25`, `run_initial_eval=True` → evals at iters 0, 25, and 49 (the final-iter override at [wmrl/train.py:505](../wmrl/train.py#L505) lands the third eval one iter early; we label it "iter 50" in the plot for readability).

In [ ]:
import dataclasses
from utils.misc import load_config
from wmrl.train import Args

yaml_cfg     = load_config("wmrl/configs/pusht.yaml")
field_names  = {f.name for f in dataclasses.fields(Args)}
args         = Args(**{k: v for k, v in yaml_cfg.items() if k in field_names})
args.config_file = "wmrl/configs/pusht.yaml"

# T4 sizing
args.num_envs        = 32
args.env_batch_num   = 4    # 4 x 8-env, for T4 GPU at colab
args.num_steps       = 4
args.mini_batch_size = 64
args.update_epochs   = 4     # 32*4/64*4 -> 8 updates per rollout
args.bc_loss_weight  = 0.0
args.use_critic      = True

# Run shape
args.seed              = 1
args.total_iterations  = 50  # just 10x8 updates, for a quick demo
args.eval_freq         = 25
args.run_initial_eval  = True
args.save_freq         = 25
args.eval_num_episodes = 50

# Wiring
args.run_dir   = "runs/colab"
args.tag       = f"pusht-corr-seed{args.seed}"
args.use_wandb = False
args.use_tb    = False
args.policy_kwargs["enable_rl_lora"] = True
args.env_kwargs.setdefault("reward_mode", "corresponding")

print(f"run_dir = {args.run_dir}/{args.tag}-...")


In [ ]:
from wmrl.train import train
success_rates = train(args)

In [ ]:
init_sr  = success_rates[0]
final_sr = max(success_rates[-2:])
print(f"initial SR  : {init_sr:.3f}")
